In [ ]:
# 2026-04-07
import pandas as pd
from time import sleep
from tqdm import tqdm
from normdata.utils import import_from_normdata
from csae_pyutils import gsheet_to_df
from apis_core.apis_metainfo.models import Collection, CollectionType, Uri
from apis_core.apis_entities.models import Person


In [ ]:
uris_to_fix = Uri.objects.filter(uri__icontains="https://d-nb.info/gnd/https://d-nb.info/gnd/")

In [ ]:
uris_to_fix.count()

In [ ]:
for x in uris_to_fix:
    new_uri = x.uri.replace("https://d-nb.info/gnd/https://d-nb.info/gnd/", "https://d-nb.info/gnd/")
    x.uri = new_uri
    x.save()

In [ ]:
df = gsheet_to_df("1AelssJSeHGE4ieZAAVS4muQn201HH4NjBq6DI1ct6-I")

In [ ]:
df

In [ ]:
col, _ = Collection.objects.get_or_create(name="DLA-Schnitzler")
domain = "dla-marbach"
col_type, _ = CollectionType.objects.get_or_create(name="Projekt")
col.description = 'To be done'
col.collection_type = col_type
col.save()

In [ ]:
print("process persons with gnd")
broken_gnd = []
for i, row in tqdm(df.iterrows()):
    tmp_uri = row["dla_url"]
    gnd = row["gnd_url"]
    gnd_uri = f"https://d-nb.info/gnd/{gnd}"
    entity = import_from_normdata(gnd_uri, "person")
    sleep(1)
    if entity:
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
        pmb_uri.entity = entity
        pmb_uri.save()
        entity.collection.add(col)
    else:
        broken_gnd.append([tmp_uri, gnd_uri])

In [ ]:
broken_gnd

In [ ]:
df = gsheet_to_df("1pxU7BWGuSPC9FN_Xf0OVV7za9PYLvjOAChYjjiygd_U")
df

In [ ]:
print("process persons without gnd")
broken_gnd = []
for i, row in tqdm(df.iterrows()):
    if pd.isna(row["dla_url"]):
        continue
    tmp_uri = row["dla_url"]
    pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_uri, domain=domain)
    entity = pmb_uri.entity
    if entity is None:
        item = {
            "name": row["surname"]
        }
        if not pd.isna(row["forename"]):
            item["first_name"] = row["forename"]
        if not pd.isna(row["birth"]):
            item["start_date_written"] = row["birth"]
        if not pd.isna(row["death"]):
            item["start_date_written"] = row["death"]
        if entity:
            pass
        
        entity = Person.objects.create(**item)
        pmb_uri.entity = entity
        pmb_uri.save()
        entity.collection.add(col)